# Week 7 – Eval: Multi-task, RAG comparison, Reasons

1. **Multi-task**: Predict category + price; report price MAE and category accuracy.
2. **RAG**: Compare **pricer only** vs **pricer + RAG** (similar products in prompt); report MAE for both.
3. **Reasons**: One-sentence "why might this cost ~$X?" for a sample.

Run from repo root. Requires GPU for inference (or run in Colab).

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
week7_dir = root / "week7" if (root / "week7" / "pricer").exists() else root
sys.path.insert(0, str(week7_dir))
contrib_dir = week7_dir / "community_contributions" / "abdussamadbello"
if contrib_dir.exists():
    sys.path.insert(0, str(contrib_dir))

import os
import re
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from pricer.evaluator import evaluate, Tester
from tqdm.notebook import tqdm
from multitask_data import build_multitask_prompt, parse_multitask_completion
from rag_retriever import build_rag_index_from_items
from reasons import get_reason

In [ ]:
load_dotenv(override=True)
if os.environ.get("HF_TOKEN"):
    login(os.environ["HF_TOKEN"], add_to_git_credential=True)

LITE_MODE = True
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"
train, val, test = Item.from_hub(dataset)
print(f"Train {len(train)}, Val {len(val)}, Test {len(test)}")

## Load model (base + PEFT adapter)

Set `ADAPTER_PATH` to your fine-tuned adapter (Hub ID or local path). If None, we define a dummy generate for structure only.

In [ ]:
ADAPTER_PATH = None  # e.g. "abdussamadbello/pricer-qlora-lite" or "/path/to/adapter"
BASE_MODEL = "meta-llama/Llama-3.2-3B"

def make_generate_fn():
    if not ADAPTER_PATH:
        return None
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel
    import torch
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    def generate(prompt: str, max_new_tokens=80) -> str:
        inp = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return generate

generate_fn = make_generate_fn()

## Multi-task predictor (category + price)

Prompt: multi-task question + summary. Parse completion to (category, price); evaluator needs price.

In [ ]:
def predict_multitask(item):
    prompt = build_multitask_prompt(item.summary or "")
    if generate_fn is None:
        return item.price  # no model: return truth for structure
    completion = generate_fn(prompt)
    cat, price = parse_multitask_completion(completion)
    return price

# Quick check
if generate_fn:
    print("Sample prediction:", predict_multitask(test[0]))

## RAG index and pricer-only vs pricer+RAG

Build index from train. Then:
- **pricer_only**: multi-task prompt with no similar products.
- **pricer_with_rag**: same prompt + "Similar products: ..." from retrieval.

In [ ]:
RAG_K = 5
rag_index = build_rag_index_from_items(train)

def prompt_with_rag(item, with_rag: bool):
    summary = item.summary or ""
    base_prompt = build_multitask_prompt(summary)
    if not with_rag:
        return base_prompt
    similar_str = rag_index.format_similar_products(summary, k=RAG_K)
    return f"{similar_str}\n\n{base_prompt}"

def pricer_only(item):
    prompt = prompt_with_rag(item, with_rag=False)
    if generate_fn is None:
        return item.price
    completion = generate_fn(prompt)
    _, price = parse_multitask_completion(completion)
    return price

def pricer_with_rag(item):
    prompt = prompt_with_rag(item, with_rag=True)
    if generate_fn is None:
        return item.price
    completion = generate_fn(prompt)
    _, price = parse_multitask_completion(completion)
    return price

## Compare pricer-only vs pricer+RAG

Run evaluate on the same test subset for both; report MAE.

In [ ]:
EVAL_SIZE = 50  # Use smaller for quick run; 200 for full comparison

if generate_fn:
    print("=== Pricer only ===")
    evaluate(pricer_only, test, size=EVAL_SIZE)
    print("=== Pricer + RAG ===")
    evaluate(pricer_with_rag, test, size=EVAL_SIZE)
else:
    print("Set ADAPTER_PATH and load model to run comparison.")

## Reasons (one-sentence why ~$X?)

For a small sample, get a reason via API (or same-model second prompt).

In [ ]:
REASONS_SAMPLE = 5
use_api = bool(os.environ.get("OPENAI_API_KEY"))

for i in range(min(REASONS_SAMPLE, len(test))):
    item = test[i]
    pred_price = pricer_only(item) if generate_fn else item.price
    pred_cat = None
    reason = get_reason(
        item.summary or "",
        pred_price,
        pred_cat,
        generate_fn=generate_fn,
        use_api=use_api,
        api_model="gpt-4o-mini",
    )
    print(f"Truth: ${item.price:.0f}  Pred: ${pred_price:.0f}  Reason: {reason}")
    print("---")